In [1]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embeddings)
retriever = db.as_retriever()

c:\EGA\code\RAG\Udemy_RAG_1\Udemy-Advanced-LangChain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
retriever.invoke("What exactly?")

[Document(id='bacf7224-6ef7-43be-a0a2-38a15781c8ba', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna'),
 Document(id='e9e172dd-3ddc-4546-a63b-9d84ed10f49f', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza')]

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser


rephrase_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language. Keep maximum inforamtion present in standalone question.

Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

REPHRASE_TEMPLATE = PromptTemplate.from_template(rephrase_template)

model11 = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0)

rephrase_chain = REPHRASE_TEMPLATE | model11 | StrOutputParser()

rephrase_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

'Does the dog really like to eat Thuna?'

In [8]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
ANSWER_PROMPT = ChatPromptTemplate.from_template(template)

In [10]:
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | ANSWER_PROMPT
    | model11
    | StrOutputParser()
)

In [11]:
final_chain = rephrase_chain | retrieval_chain

In [13]:
final_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)

'Based on the provided context, the dog loves to eat pizza. There is no information suggesting that the dog does not like to eat anything.'

### Chat with returning documents

In [20]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

retrieved_documents = RunnableParallel(
    {"docs": retriever, 
     "question": RunnablePassthrough()
    }
)

final_inputs = {
    "context": lambda x: "\n".join(doc.page_content for doc in x["docs"]),
    "question": lambda x: x["question"],
}
answer = {
    "answer": final_inputs | ANSWER_PROMPT | model11 | StrOutputParser(),
    "docs": lambda x: x["docs"],
}

final_chain = rephrase_chain | retrieved_documents | answer

In [21]:
result = final_chain.invoke(
    {
        "question": "Not really?",
        "chat_history": [
            HumanMessage(content="What does the dog like to eat?"),
            AIMessage(content="Thuna!"),
        ],
    }
)
print(result)

{'answer': 'Based on the context, the dog loves to eat pizza, so it does like to eat something. Therefore, the dog does not really not like to eat anything.', 'docs': [Document(id='e9e172dd-3ddc-4546-a63b-9d84ed10f49f', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'), Document(id='bacf7224-6ef7-43be-a0a2-38a15781c8ba', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]}


In [22]:
result["answer"]

'Based on the context, the dog loves to eat pizza, so it does like to eat something. Therefore, the dog does not really not like to eat anything.'

In [23]:
result["docs"]

[Document(id='e9e172dd-3ddc-4546-a63b-9d84ed10f49f', metadata={'source': 'animal.txt'}, page_content='the dog loves to eat pizza'),
 Document(id='bacf7224-6ef7-43be-a0a2-38a15781c8ba', metadata={'source': 'animal.txt'}, page_content='the cat loves to eat lasagna')]